# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the [FAIR² dataset](https://doi.org/10.71728/senscience.y7m0-f273) using the `mlcroissant` library. We demonstrate how to access Croissant metadata, inspect record sets, extract data by entity `@id`, and perform basic exploratory analysis and visualization.

### Dataset Source
This dataset is described via a Croissant schema, accessible at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure the mlcroissant library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset Croissant URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Name: {metadata.name}\n\nDescription: {metadata.description}\n")

# For reproducibility later
SEED = 42
np.random.seed(SEED)

## 2. Data Overview
Review available record sets, fields, and their `@id` values using the `mlcroissant` API.

In [ ]:
# List all record sets, fields and columns by their '@id'

print("Available Record Sets:")
record_sets = list(metadata.record_sets)
for i, rs in enumerate(record_sets):
    print(f"  {i + 1}. @id: {rs['@id']}")

# For each record set, list its fields by @id
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for fld in fields:
        if isinstance(fld, dict):
            print(f"    - @id: {fld.get('@id', str(fld))} (name: {fld.get('name', '<unnamed>')})")
        else:
            print(f"    - {fld}")

## 3. Data Extraction
Load data from the record sets using the `@id` values. Data is extracted into pandas DataFrame objects for inspection and analysis.

In [ ]:
# Extract all available record sets by @id. You'll need to fill in or check these IDs from above:
record_set_ids = [rs['@id'] for rs in metadata.record_sets]

# Display list of record set IDs
print("Record Set IDs to extract:", record_set_ids)

# Extract each record set to dataframe
dfs = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records for record_set '@id': {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dfs[record_set_id] = pd.DataFrame(records)
        print(f"  Columns: {dfs[record_set_id].columns.tolist()}")
        display(dfs[record_set_id].head(3))
    else:
        print("  No records found.")

# For further analysis, select the first available record set if exists:
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"\nUsing main record set '@id': {main_record_set_id}")
    main_df = dfs.get(main_record_set_id)
    if main_df is not None:
        print(f"Available columns: {main_df.columns.tolist()}")
        main_df.head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing operations: filtering on numeric fields, normalizing values, and grouping by categorical variables, using `@id` for referencing.

In [ ]:
# If main_df is not None and contains numeric fields, proceed with EDA
if 'main_df' in locals() and main_df is not None and not main_df.empty:
    # Identify numeric fields by checking datatypes
    numeric_fields = main_df.select_dtypes(include=[np.number]).columns.tolist()
    
    if numeric_fields:
        numeric_field_id = numeric_fields[0]  # Choose the first numeric field
        print(f"Example numeric field selected: '{numeric_field_id}'\n")
        
        # Demonstrate threshold filtering
        threshold = main_df[numeric_field_id].mean()  # Use mean as example threshold
        filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with '{numeric_field_id}' > {threshold:.2f}:")
        display(filtered_df.head())
        
        # Normalize the selected numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())
        
        # Choose a string/categorical field to group by (if any exists)
        group_fields = main_df.select_dtypes(include=[object]).columns.tolist()
        if group_fields:
            group_field_id = group_fields[0]
            print(f"\nGrouping by field: '{group_field_id}'\n")
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean '{numeric_field_id}' by '{group_field_id}':")
            display(grouped.head())
            # Save for plot below
            grouped_example = grouped
    else:
        print("No numeric fields available in main_df for analysis.")
else:
    print("No available main DataFrame for EDA. Please check that your dataset is loaded and record sets contain records.")

## 5. Visualization

Create simple plots to visualize distributions or relationships. The plots use field `@id` names, as per Croissant specification.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Demonstrate a histogram of the main numeric field if present
if 'main_df' in locals() and main_df is not None and not main_df.empty and 'numeric_field_id' in locals():
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field_id], bins=20, kde=True, color='dodgerblue')
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.tight_layout()
    plt.show()

# If group analysis done, make a bar plot
if 'grouped_example' in locals():
    plt.figure(figsize=(10,4))
    sns.barplot(data=grouped_example, x=group_field_id, y=numeric_field_id, color='lightslategrey')
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion

This notebook demonstrated how to utilize the Croissant `@id` referencing mechanism—using the `mlcroissant` library—to:
- Load a FAIR²-compliant dataset from schema
- Explore record set details by their `@id`
- Extract data to pandas DataFrames using Croissant entity `@id`
- Apply standard numerical EDA transformations using field `@id`s
- Visualize variable distributions

For full reproducibility and to extend the exploration, refer to the fields and record set `@id` values output in the *Data Overview* step. As this dataset offers regression outputs on rangeland management adoption predictors, further statistical or ML analysis can be performed using the extracted data.